# Generate OLD/WEAK-generator reviews (GPT-2-era) across all domains

Fixes a generalization gap in the broadened CG/OR detector: the GPT-2-era / old-synthetic
generators (Salminen GPT-2, sutro, Kenshiii) lived **only in the products domain**, with no
siblings — so leave-one-out evals showed salminen_cg LOGO 0.07 and products LOD 0.34 (the
model couldn't recognise old-generation text it hadn't seen).

This adds an **"old/weak LLM" family spread across all 6 domains** using small base models —
**distilgpt2 · gpt2 · gpt2-medium · OPT-350m** (open, tiny, fast, no API). They are NOT
instruction-tuned, so we seed each with a natural review-opening prefix and let them continue
(their rambly, repetitive output IS the old-generation distribution we want to cover).
Length is controlled per-domain to match the real review length distribution.

### Instructions
1. **Runtime -> GPU** (these are tiny; any GPU is plenty, even T4 with LOAD_4BIT=False).
2. Run all cells. Download `ai_oldgen_reviews_open.csv` -> put in `data/` and tell the assistant.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import os, csv, random, gc, shutil
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

SEED=42; random.seed(SEED)
MODELS=[("distilgpt2","distilgpt2"),("gpt2","gpt2"),
        ("gpt2-medium","gpt2med"),("facebook/opt-350m","opt350")]   # old/weak families
PER_DOMAIN_PER_MODEL=400      # 6 domains x 4 models x 400 ~ 9,600 (tiny models -> fast)
BATCH=64
OUT="ai_oldgen_reviews_open.csv"

# bucket -> max_new_tokens (base models: ~tokens>words, so a bit higher than word targets)
LENGTH_BUCKETS={"tiny":30,"short":70,"medium":130,"long":220}
DOMAIN_BUCKETS={
    "apps":        {"tiny":0.55,"short":0.30,"medium":0.13,"long":0.02},
    "games":       {"tiny":0.45,"short":0.28,"medium":0.20,"long":0.07},
    "books":       {"tiny":0.10,"short":0.25,"medium":0.30,"long":0.35},
    "restaurants": {"tiny":0.10,"short":0.25,"medium":0.40,"long":0.25},
    "products":    {"tiny":0.08,"short":0.32,"medium":0.45,"long":0.15},
    "movies":      {"tiny":0.02,"short":0.08,"medium":0.30,"long":0.60},
}
# review-opening prefixes to seed the (non-instruct) base models
PREFIXES={
 "products":["I bought this product and","Honestly, this item","I ordered this and","After using this for a while,","This product is","I was looking for something like this and"],
 "movies":["I watched this movie and","This film was","I saw this last night and","The movie was","Honestly the film","I went to see this movie and"],
 "restaurants":["I went to this restaurant and","The food here was","We ate at this place and","This restaurant is","I had dinner here and","Stopped by for lunch and"],
 "apps":["I downloaded this app and","This app is","After using this app for a few days,","I have been using this app and","The app"],
 "books":["I read this book and","This book was","The story was","I just finished this book and","Honestly the writing"],
 "games":["I played this game and","This game is","The gameplay was","After playing for a few hours,","I picked up this game and"],
}

In [ ]:
def pick_bucket(domain):
    b=list(DOMAIN_BUCKETS[domain]); w=list(DOMAIN_BUCKETS[domain].values())
    return random.choices(b,weights=w,k=1)[0]
def build_prompt(domain):
    bucket=pick_bucket(domain)
    prefix=random.choice(PREFIXES[domain])
    return prefix, bucket, LENGTH_BUCKETS[bucket]

In [ ]:
have={fam:0 for _,fam in MODELS}
if os.path.exists(OUT):
    import pandas as pd
    try:
        for s,c in pd.read_csv(OUT)["source"].value_counts().items():
            if s in have: have[s]=int(c)
    except Exception: pass
print("existing per generator:", have)
new=not os.path.exists(OUT)
f=open(OUT,"a",newline="",encoding="utf-8"); w=csv.writer(f)
if new: w.writerow(["text","domain","source","model_id","length_bucket"])
TARGET=PER_DOMAIN_PER_MODEL*len(DOMAIN_BUCKETS)

for model_id,fam in MODELS:
    if have[fam]>=TARGET: print(f"== {fam}: has {have[fam]}, skip =="); continue
    print(f"\n=== loading {model_id} ===")
    try:
        # explicit torch load -> no `framework` kwarg leak, no TF/Keras inference path
        mdl=AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)
        tok=AutoTokenizer.from_pretrained(model_id)
        if tok.pad_token_id is None: tok.pad_token_id=tok.eos_token_id
        tok.padding_side="left"
        pipe=pipeline("text-generation", model=mdl, tokenizer=tok,
                      device=0 if torch.cuda.is_available() else -1)
    except Exception as e:
        print("  load failed, skipping:", e); continue
    made=0
    for domain in DOMAIN_BUCKETS:
        # group by bucket so each batch shares one max_new_tokens
        from collections import defaultdict
        by_b=defaultdict(int)
        for _ in range(PER_DOMAIN_PER_MODEL): by_b[pick_bucket(domain)]+=1
        for bucket,cnt in by_b.items():
            mt=LENGTH_BUCKETS[bucket]
            prompts=[random.choice(PREFIXES[domain]) for _ in range(cnt)]
            for i in range(0,len(prompts),BATCH):
                batch=prompts[i:i+BATCH]
                try:
                    outs=pipe(batch,max_new_tokens=mt,do_sample=True,temperature=0.95,top_p=0.95,
                              batch_size=len(batch),pad_token_id=tok.pad_token_id,truncation=True)
                except Exception as e:
                    print("  gen error:",e); continue
                for o in outs:
                    rec=o[0] if isinstance(o,list) else o
                    text=str(rec["generated_text"]).strip().replace("\n"," ")
                    if len(text.split())>=4:
                        w.writerow([text,domain,fam,model_id,bucket]); made+=1
        f.flush(); print(f"  {fam} {domain}: total {made}")
    print(f"== {fam}: +{made} ==")
    del pipe; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--"+model_id.replace("/","--")),ignore_errors=True)
f.close(); print("done ->",OUT)

In [ ]:
import pandas as pd
df=pd.read_csv(OUT); wl=df["text"].str.split().str.len()
print("total:",len(df),"| per generator:",df["source"].value_counts().to_dict())
print("per domain:",df["domain"].value_counts().to_dict())
print("median words:",int(wl.median()))
from google.colab import files
files.download(OUT)